# 04 — Cluster profiles and the PM4 signature

This notebook compares AML clusters with cell groups from the other studies using relative expression
profiles.
It also finds genes that distinguish PM4 from other AML clusters in both leukemia arms.

Steps:
1. Calculate relative expression profiles using the integration genes.
2. Compare AML profiles with the Yoon and published cell groups by Spearman correlation.
3. Find PM4 markers in pooled AML cells and retain those confirmed within both leukemia arms.

Outputs: two profile-correlation tables and `pm4_signature_genes.tsv` in `cluster_profiles/`.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scipy.stats import spearmanr

from settings import AML_DATASET, ATLAS_FILE, HVG_TABLE, RESULTS_DIR

## Settings

In [ ]:
OUTPUT_DIR = RESULTS_DIR / "cluster_profiles"  # tables used by notebook 06
MIN_GROUP_CELLS = 30                          # minimum size for profiles and marker-test groups
MARKER_MIN_LOG2FC = 1                         # at least a twofold increase in PM4
MARKER_MIN_DETECTION = 0.10                   # detected in at least 10% of PM4 cells
MARKER_MAX_ADJUSTED_P = 0.05                  # multiple-testing-adjusted p-value cutoff

LEUKEMIA_ARMS = ["AML1-ETO", "RasV12"]         # arms that must both confirm the signature
SIGNATURE_CLUSTER = "PM4"                     # cluster whose markers are tested

LABEL_SCHEMES = {"yoon_labels": "yoon_subtype", "published_labels": "published_label"}

## Load expression and group labels
Expression is log1p counts per 10,000. Published labels identify the AML clusters.
Groups need at least 30 cells; smaller clusters still contribute to dataset means.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

atlas = ad.read_h5ad(ATLAS_FILE)
is_aml = atlas.obs["dataset"].eq(AML_DATASET).to_numpy()
aml = atlas[is_aml].copy()
aml.obs["published_label"] = aml.obs["published_label"].astype(str)

cluster_sizes = aml.obs["published_label"].value_counts()
display(pd.crosstab(aml.obs["published_label"], aml.obs["condition"]))

## Select the integration genes
The 2,000 integration genes stay in atlas order so that every profile uses the same coordinates.
A group's relative profile is its mean expression minus its dataset's mean expression. Positive values mean
higher expression than the dataset average; this highlights each group's pattern rather than study-wide
levels.

In [ ]:
hvg_ids = pd.read_csv(HVG_TABLE, sep="\t", index_col=0).index
genes = atlas.var_names[atlas.var_names.isin(hvg_ids)]
gene_positions = atlas.var_names.get_indexer(genes)


def relative_profiles(data, label_column):
    """Mean log1p expression per group, relative to the dataset mean."""
    expression = data.X[:, gene_positions]
    overall = np.asarray(expression.mean(axis=0)).ravel()
    profiles = {}

    for label, cells in data.obs.groupby(label_column, observed=True).groups.items():

        if len(cells) >= MIN_GROUP_CELLS:
            rows = data.obs_names.get_indexer(cells)
            profiles[label] = np.asarray(expression[rows].mean(axis=0)).ravel() - overall

    return pd.DataFrame(profiles, index=genes)

## Compare AML clusters with each reference group
Spearman correlation compares the ranks of the relative expression values.
Each reference dataset is centered separately, using only cells with a label in that scheme.

In [ ]:
aml_profiles = relative_profiles(aml, "published_label")
datasets = sorted(atlas.obs.loc[~is_aml, "dataset"].astype(str).unique())


def correlate(label_column):
    """Correlate every AML profile with every dataset and label profile."""
    reference_profiles = []

    for dataset in datasets:
        has_label = atlas.obs[label_column].notna().to_numpy()
        in_dataset = atlas.obs["dataset"].eq(dataset).to_numpy()

        profiles = relative_profiles(atlas[in_dataset & has_label], label_column)
        profiles.columns = pd.MultiIndex.from_product([[dataset], profiles.columns.astype(str)])

        reference_profiles.append(profiles)

    reference_profiles = pd.concat(reference_profiles, axis=1)
    correlations = {}

    for reference in reference_profiles:
        values = []

        for cluster in aml_profiles:
            correlation = spearmanr(aml_profiles[cluster], reference_profiles[reference]).statistic
            values.append(correlation)

        correlations[reference] = values

    table = pd.DataFrame(correlations, index=aml_profiles.columns)
    table.columns = pd.MultiIndex.from_tuples(table.columns, names=["dataset", "label"])

    return table

## Save correlations for both label schemes
Yoon labels give a common vocabulary across studies; published labels retain each study's own groups.
The two header rows identify the dataset and label of each reference profile.

In [ ]:
for name, label_column in LABEL_SCHEMES.items():
    table = correlate(label_column)
    table.to_csv(OUTPUT_DIR / f"profile_correlation_{name}.tsv", sep="\t")
    display(table.iloc[:5, :5])

## Find PM4 markers across the pooled AML cells
PM4 is compared with the other AML clusters that have at least 30 cells overall.
Markers need log2 fold change ≥ 1, detection in ≥ 10% of PM4 cells, and adjusted p ≤ 0.05.

In [ ]:
tested_clusters = sorted(cluster_sizes[cluster_sizes >= MIN_GROUP_CELLS].index)
pooled = aml[aml.obs["published_label"].isin(tested_clusters)].copy()
sc.tl.rank_genes_groups(
    pooled,
    "published_label",
    groups=[SIGNATURE_CLUSTER],
    reference="rest",
    method="wilcoxon",
    use_raw=False,
    pts=True,
)

pooled_markers = sc.get.rank_genes_groups_df(pooled, group=SIGNATURE_CLUSTER).set_index("names")
pooled_markers["symbol"] = pooled_markers.index.map(atlas.var["symbol"])


def marker_mask(table):
    """Genes meeting the fold change, detection and adjusted p-value cutoffs."""

    return (
        (table["logfoldchanges"] >= MARKER_MIN_LOG2FC)
        & (table["pct_nz_group"] >= MARKER_MIN_DETECTION)
        & (table["pvals_adj"] <= MARKER_MAX_ADJUSTED_P)
    )

## Repeat the PM4 comparison inside each leukemia arm
Each arm is restricted to clusters with at least 30 cells in that arm.
Testing PM4 against other clusters inside each arm avoids calling a gene a PM4 marker just because it is
higher
in one leukemia arm. A signature gene must meet the same cutoffs in the pooled test and in both arms.

In [ ]:
columns = ["logfoldchanges", "pct_nz_group"]
signature = pooled_markers[["symbol"]].join(pooled_markers[columns].add_prefix("pooled_"))
is_signature = marker_mask(pooled_markers)

for arm in LEUKEMIA_ARMS:
    in_arm = aml.obs["condition"].eq(arm).to_numpy()
    sizes = aml.obs.loc[in_arm, "published_label"].value_counts()
    groups = sorted(sizes[sizes >= MIN_GROUP_CELLS].index)
    in_group = aml.obs["published_label"].isin(groups).to_numpy()

    arm_data = aml[in_arm & in_group].copy()

    sc.tl.rank_genes_groups(
        arm_data,
        "published_label",
        groups=[SIGNATURE_CLUSTER],
        reference="rest",
        method="wilcoxon",
        use_raw=False,
        pts=True,
    )

    markers = sc.get.rank_genes_groups_df(arm_data, group=SIGNATURE_CLUSTER).set_index("names")
    signature = signature.join(markers[columns].add_prefix(f"{arm}_"))

    is_signature = is_signature & marker_mask(markers).reindex(signature.index)

## Save the PM4 signature in pooled ranking order
Only genes that pass all three tests are kept, in their pooled ranking order.
This order is also used when scoring the signature in the figures.

In [ ]:
signature = signature.loc[is_signature]
signature.index.name = "gene_id"
signature.to_csv(OUTPUT_DIR / "pm4_signature_genes.tsv", sep="\t")
display(signature.head(10))

print(f"{len(signature)} PM4 signature genes: {', '.join(signature['symbol'])}")